Importação de Bibliotecas

In [ ]:
import pandas as pd
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt 
import stats

# Análise da Base

In [3]:
df = pd.read_excel("documentos/base-antecipacao-AULA.xlsx")
df.columns = df.columns.str.strip()

In [18]:
df.describe()

,data_operacao,score_credito,prazo_dias,volume_solicitado,taxa_pct,lucro_operacao,inadimplencia,horas_analise_antes,horas_analise_depois
count,720,715.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000,720.000000
mean,2025-07-07 18:51:59.999999744,612.474126,41.291667,9432.833222,2.770556,216.409264,0.127778,6.125472,5.784778
min,2025-01-07 00:00:00,300.000000,30.000000,190.920000,1.270000,3.460000,0.000000,1.500000,0.500000
25%,2025-04-05 00:00:00,543.000000,30.000000,2310.277500,2.379750,51.567500,0.000000,4.965000,4.370000
50%,2025-07-10 00:00:00,613.000000,45.000000,4877.300000,2.757000,114.770000,0.000000,6.100000,5.810000
75%,2025-10-07 06:00:00,676.000000,45.000000,10485.882500,3.160250,255.732500,0.000000,7.270000,7.140000
max,2025-12-31 00:00:00,850.000000,60.000000,193384.680000,4.312000,3260.880000,1.000000,12.970000,13.200000
std,NaN,95.926897,11.840311,14486.393334,0.542879,328.241850,0.334074,1.725622,2.145163


Quantas linhas e colunas? Onde estão os 5 ausentes?

In [9]:
print(f"Quantidade de Linhas: {len(df)}")
print(f'Quantidade de Colunas: {len(df.columns)}\n')

# Colunas com valores nulos
print("Colunas com Valores Nulos:\n")
print(df.isna().sum()[df.isna().sum() > 0])

Quantidade de Linhas: 720
Quantidade de Colunas: 13

Colunas com Valores Nulos:

score_credito    5
dtype: int64


In [10]:
df = df.drop(columns=['id_operacao'])

## Análise de Valores distintos

In [11]:
df['segmento'].value_counts()

segmento
Industria    240
Comercio     240
Servicos     240
Name: count, dtype: int64

In [12]:
df['porte'].value_counts()

porte
Micro      372
Pequena    248
Media      100
Name: count, dtype: int64

In [13]:
df['canal'].value_counts()

canal
Gerente           360
Plataforma Web    360
Name: count, dtype: int64

In [14]:
df['inadimplencia'].value_counts()

inadimplencia
0    628
1     92
Name: count, dtype: int64

In [16]:
# Análise de Colunas com valores repetidos

for x in df.columns:
    qtd_repetidos = df[x].duplicated().sum()
    print(f'{x} | Valores duplicados: {qtd_repetidos}\n')

data_operacao | Valores duplicados: 411

segmento | Valores duplicados: 717

canal | Valores duplicados: 718

porte | Valores duplicados: 717

score_credito | Valores duplicados: 405

prazo_dias | Valores duplicados: 717

volume_solicitado | Valores duplicados: 0

taxa_pct | Valores duplicados: 113

lucro_operacao | Valores duplicados: 11

inadimplencia | Valores duplicados: 718

horas_analise_antes | Valores duplicados: 282

horas_analise_depois | Valores duplicados: 250



## Adaptação de dados

In [19]:
df['valor_financeiro'] = df['taxa_pct']/100 * df['volume_solicitado']

In [20]:
df[['volume_solicitado', 'taxa_pct', 'valor_financeiro']]

,volume_solicitado,taxa_pct,valor_financeiro
0,2917.51,2.472,72.120847
1,7123.80,2.508,178.664904
2,2076.15,2.318,48.125157
3,553.81,2.977,16.486924
4,3855.99,3.404,131.257900
...,...,...,...
715,15038.07,2.045,307.528531
716,741.29,2.587,19.177172
717,5704.90,3.582,204.349518
718,7936.24,2.865,227.373276


In [21]:
df['score_credito'] = df['score_credito'].fillna(0)

# Hipóteses

In [ ]:
alpha = 0.05

# Premissas

score_credito é normal? volume_solicitado é normal? As variâncias são iguais?

Para Shapiro-Wilk e Levene, consideramos:

- *p > 0,05:* não rejeitamos a premissa.
- *p ≤ 0,05:* há evidência contra a premissa.

In [ ]:
# Criação de uma figura lado a lado com apenas histogramas KDE

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

score = df["score_credito"].dropna()
volume = df["volume_solicitado"].dropna()

# 1. Histograma com KDE: score_credito
sns.histplot(score, kde=True, color='skyblue', bins=30, ax=axes[0])
axes[0].set_title("Distribuição: score_credito (Simétrica / Sino)")
axes[0].set_xlabel('Score de Crédito')
axes[0].set_ylabel('Frequência')

# 2. Histograma com KDE: volume_solicitado
sns.histplot(volume, kde=True, color='salmon', bins=30, ax=axes[1])
axes[1].set_title("Distribuição: volume_solicitado (Assimétrica / Gama)")
axes[1].set_xlabel('Volume Solicitado')
axes[1].set_ylabel('Frequência')

plt.tight_layout()
plt.show()

In [ ]:
# Normalidade
score = df["score_credito"].dropna()
volume = df["volume_solicitado"].dropna()

shapiro_score = stats.shapiro(score)
shapiro_volume = stats.shapiro(volume)

print("Shapiro-Wilk — score_credito")
print(f"W = {shapiro_score.statistic:.6f}; p = {shapiro_score.pvalue:.6g}")

print("\nShapiro-Wilk — volume_solicitado")
print(f"W = {shapiro_volume.statistic:.6f}; p = {shapiro_volume.pvalue:.6g}")

print("\nInterpretação:")
print("score_credito:", "compatível com normalidade." if shapiro_score.pvalue > alpha else "não compatível com normalidade.")
print("volume_solicitado:", "compatível com normalidade." if shapiro_volume.pvalue > alpha else "não compatível com normalidade.")

In [ ]:
# Homogeneidade das variâncias
taxa_gerente = df.loc[df["canal"] == "Gerente", "taxa_pct"]
taxa_web = df.loc[df["canal"] == "Plataforma Web", "taxa_pct"]

score_por_segmento = [
    grupo["score_credito"].dropna()
    for _, grupo in df.groupby("segmento")
]

levene_taxa = stats.levene(taxa_gerente, taxa_web)
levene_score = stats.levene(*score_por_segmento)

print("Levene — taxa_pct por canal")
print(f"W = {levene_taxa.statistic:.6f}; p = {levene_taxa.pvalue:.6g}")

print("\nLevene — score_credito por segmento")
print(f"W = {levene_score.statistic:.6f}; p = {levene_score.pvalue:.6g}")

# Análise da Taxa Média

A taxa média é diferente entre Gerente e Plataforma Web?

# Avaliação do Score Médio

O score médio é diferente entre os 3 segmentos? Quais pares diferem?

In [ ]:
segmentos = pd.unique(df['segmento'])

# Redução Horas Análise

O novo sistema reduziu as horas de análise? De quanto foi o ganho?

# Score vs Taxa

Score e taxa andam juntos? Quanto de taxa vale 100 pontos de score?

In [ ]:
corr = stats.pearsonr(score, df.loc[score.index, "taxa_pct"])

X = sm.add_constant(score)
y = df.loc[score.index, "taxa_pct"]
modelo = sm.OLS(y, X).fit()

beta = modelo.params["score_credito"]
ic_beta = modelo.conf_int().loc["score_credito"]

efeito_100 = beta * 100
ic_efeito_100 = (ic_beta[0] * 100, ic_beta[1] * 100)

print(f"Pearson r = {corr.statistic:.3f}")
print(f"p = {corr.pvalue:.6g}")
print(f"n = {len(score)}")

print("\nModelo:")
print(f"taxa = {modelo.params['const']:.4f} + ({beta:.6f}) × score")

print(f"\nEfeito estimado para +100 pontos: {efeito_100:.3f} p.p.")
print(f"IC95%: [{ic_efeito_100[0]:.3f}, {ic_efeito_100[1]:.3f}] p.p.")

print("\nConclusão:",
      "há associação negativa estatisticamente significativa."
      if corr.pvalue < alpha
      else "não há evidência de associação linear significativa.")

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(score, y, alpha=0.5)
x_line = np.linspace(score.min(), score.max(), 100)
y_line = modelo.params["const"] + beta * x_line
plt.plot(x_line, y_line)
plt.title("Relação entre score de crédito e taxa")
plt.xlabel("Score de crédito")
plt.ylabel("Taxa (%)")
plt.show()

Os dados demonstram uma correlação negativa moderada a forte ($r = -0,554$, $p = 1,09944 \times 10^{-58}$, $n = 715$), indicando que clientes com melhor score de crédito tendem a pagar taxas menores. 

Para cada acréscimo de 100 pontos no score de crédito, a taxa cobrada diminui em média 0,314 p.p. (com um intervalo de confiança de 95% variando entre $-0,349$ e $-0,279$ p.p.).

Natureza da Relação: Ressalta-se que a análise comprova uma relação de associação, e não necessariamente de causalidade direta.

SIM. Clientes com score melhor tendem a pagar taxa menor. A cada 100 pontos adicionais de score, a taxa varia em -0.314 p.p. em média (IC95% -0.349 a -0.279; n=715; p=1.09944e-58).

Estimativa: Redução de $-0,314$ p.p. na taxa  Intervalo: IC 95% $[-0,349, -0,279]$ p.p.  n: 720

# Dependencia da Indadimplência

A inadimplência depende do canal? E do porte?

# Intervalo e relatório

Reporte cada resultado como estimativa + intervalo + n.